# M2 (continued) — finish the run the clock cut off

The first M2 run stopped at epoch 2.76 of 3 on its 200-minute budget, having seen 608k
examples against B3's 832k and T2's 1,056k. What makes that the headline rather than a
footnote is the slope at the stopping point:

| run | eval_loss at stop | loss fall per epoch, last epoch |
|---|---:|---:|
| B3 | 0.7671 | 0.0077 |
| T2 | 0.6796 | 0.0049 |
| M2 | 0.7468 | **0.0332** |

M2 was cut off while still improving seven times faster than T2 was at its own finish, so
its 2.0–3.6 point deficit against T2 on every field is what an unfinished run looks like,
not what the schema-prefix scheme is worth. This kernel resumes from the exact checkpoint —
optimizer state and step count included — and spends what is left of the week's GPU hours
on the same corpus.

**Budget: about 2 GPU hours left.** 75 minutes of training (≈ +1.1 epoch, 240k more
examples), then the same full-test evaluation as before, so the numbers stay directly
comparable to B3, T1, T2 and M2's first stop.

`--ignore-data-skip` matters here: without it Trainer replays every batch already consumed
before doing any work, and at ~19k steps in that would eat a large share of the 75 minutes.

**Before running:** Settings → **Internet** on, **GPU T4 x2** on. Run as **Save & Run All (Commit)**.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available(), "| devices:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  Device {i}:", torch.cuda.get_device_name(i))

In [ ]:
import os
if not os.path.isdir("retro-planner"):
    !git clone https://github.com/oleh-kuzmenko/retro-planner.git
%cd retro-planner

In [ ]:
%pip install -q -e ".[local-models,indexing]"

In [ ]:
import glob, json

train_file = next(glob.iglob("/kaggle/input/**/conditions_train.jsonl", recursive=True))
val_file = next(glob.iglob("/kaggle/input/**/conditions_val.jsonl", recursive=True))
test_file = next(glob.iglob("/kaggle/input/**/conditions_test_clean.jsonl", recursive=True))
for path in (train_file, val_file, test_file):
    print(path, sum(1 for _ in open(path)), "rows")

resume_dir = next(glob.iglob("/kaggle/input/**/latest_checkpoint", recursive=True))
state = json.load(open(f"{resume_dir}/trainer_state.json"))
print(f"\nresuming from {resume_dir}")
print(f"  epoch {state['epoch']:.2f} | global step {state['global_step']} | best eval_loss {state['best_metric']:.4f}")
assert state["epoch"] > 2.5, "this is not the checkpoint the first M2 run left behind"

condition_fields = "reagents,solvent,catalyst,temperature_celsius"
always_fields = "solvent,temperature_celsius"
output_dir = "/kaggle/working/model2_conditions_mixed"
time_budget_minutes = 75

In [ ]:
import os

os.makedirs(output_dir, exist_ok=True)
log_path = f"{output_dir}/train.log"

# num-train-epochs is raised to 4.5 so the schedule has room past the resume point; the
# time budget, not the epoch count, is what will stop this run.
!torchrun --nproc_per_node=2 scripts/train_conditions_model.py \
    --base-model "t5-small" \
    --resume-from-checkpoint "{resume_dir}" \
    --ignore-data-skip \
    --train-file "{train_file}" \
    --val-file "{val_file}" \
    --output-dir "{output_dir}" \
    --local-work-dir /kaggle/temp/local_model2_mixed \
    --target-format compact \
    --condition-fields "{condition_fields}" \
    --always-fields "{always_fields}" \
    --max-source-length 256 \
    --max-target-length 256 \
    --learning-rate 5e-4 \
    --num-train-epochs 4.5 \
    --time-budget-minutes {time_budget_minutes} \
    > "{log_path}" 2>&1
print("training done; tail of log:")
!tail -5 "{log_path}"

In [ ]:
import json

state = json.load(open(f"{output_dir}/latest_checkpoint/trainer_state.json"))
points = [(h["epoch"], h["eval_loss"]) for h in state["log_history"] if "eval_loss" in h]
for epoch, loss in points[:: max(1, len(points) // 12)]:
    print(f"  epoch {epoch:5.2f}  eval_loss {loss:.4f}")
print(f"  reached epoch {points[-1][0]:.2f}, best {state.get('best_metric'):.4f}")

# The question this run exists to answer: is the curve still as steep as it was at 2.76?
tail = [p for p in points if p[0] >= points[-1][0] - 1]
if len(tail) > 1:
    slope = (tail[0][1] - tail[-1][1]) / (tail[-1][0] - tail[0][0])
    print(f"  loss fall over the last epoch: {slope:.4f} (was 0.0332 at the first stop, "
          f"T2 finished at 0.0049)")

In [ ]:
!python scripts/evaluate_conditions_model_topk.py \
    --model-dir "{output_dir}/final" \
    --test-file "{test_file}" \
    --num-beams 10 --batch-size 32 --device cuda \
    --max-source-length 256 --max-target-length 256 \
    --output "/kaggle/working/M2b_conditions_mixed_clean_topk.json"

In [ ]:
import json
summary = json.load(open("/kaggle/working/M2b_conditions_mixed_clean_topk.json"))["summary"]
print(json.dumps(summary, indent=2))

# Same four rows as before, so the effect of the extra epoch is readable directly.
previous = {
    "reagents_exact_match_top5": ("M2@2.76 0.169", "B3 0.215", "T2 0.184"),
    "solvent_exact_match_top5": ("M2@2.76 0.250", "B3 0.338", "T2 0.269"),
    "catalyst_exact_match_top5": ("M2@2.76 0.160", "B3 0.288", "T2 0.176"),
    "temperature_celsius_within_tol_top5": ("M2@2.76 0.539", "B3 0.157", "T2 0.546"),
}
print("\nfield                                   now      before / B3 / T2")
for key, refs in previous.items():
    print(f"{key:38} {summary.get(key, 0):.3f}    " + " / ".join(refs))